In [17]:
import pandas as pd

# Load the CSV file into a DataFrame
# Thank the autocomplete
df = pd.read_csv('Server_Log.csv')
df.columns = df.columns.str.strip()
df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce')
df.set_index('Timestamp', inplace=True)

print(df.head())

                               Server CPU Usage Memory Usage Disk Usage
Timestamp                                                              
2026-03-02 14:20:03.179135   Server C       37%      2560 MB      26 GB
2026-03-02 14:20:03.180243   Server A       23%      3328 MB      11 GB
2026-03-02 14:20:03.181788   Server B       47%      3072 MB      26 GB
2026-03-02 14:20:03.183497   Server C       62%      3840 MB      40 GB
2026-03-02 14:20:03.185139   Server C        8%      2048 MB      67 GB


In [18]:
# Clean and convert data types from String to Int
df["CPU Usage"] = df["CPU Usage"].str.lstrip().str.rstrip('%').astype(int)
df["Memory Usage"] = df["Memory Usage"].str.lstrip().str.rstrip(' MB').astype(int)
df["Disk Usage"] = df["Disk Usage"].str.lstrip().str.rstrip(' GB').astype(int)
df['Server'] = df['Server'].str.strip()
print(df.head())

                              Server  CPU Usage  Memory Usage  Disk Usage
Timestamp                                                                
2026-03-02 14:20:03.179135  Server C         37          2560          26
2026-03-02 14:20:03.180243  Server A         23          3328          11
2026-03-02 14:20:03.181788  Server B         47          3072          26
2026-03-02 14:20:03.183497  Server C         62          3840          40
2026-03-02 14:20:03.185139  Server C          8          2048          67


In [19]:
#Identify high usage entries
#Criteria: CPU Usage > 80%, Memory Usage > 8000 MB, Disk Usage > 500 GB
high_usage = df[(df['CPU Usage'] > 80) | (df['Memory Usage'] > 8000) | (df['Disk Usage'] > 500)]
print(high_usage)

                              Server  CPU Usage  Memory Usage  Disk Usage
Timestamp                                                                
2026-03-02 14:20:03.188754  Server A         89           768          58
2026-03-02 14:20:03.209046  Server C         85          2304          79
2026-03-02 14:20:03.217501  Server A         97          2560          84
2026-03-02 14:20:03.237210  Server B         92          2816         100
2026-03-02 14:20:03.239006  Server B         99          2304          22
...                              ...        ...           ...         ...
2026-03-02 14:20:20.810405  Server A         86          3328          46
2026-03-02 14:20:20.817147  Server B         99          2560          60
2026-03-02 14:20:20.829279  Server A         81          1024          45
2026-03-02 14:20:20.835918  Server A         90          2816          26
2026-03-02 14:20:20.856775  Server A         93          1024          47

[1967 rows x 4 columns]


In [20]:
#Split the data per server
#print to confirm the split
Server_High_Usage = {name: group for name, group in high_usage.groupby(high_usage["Server"])}
print(Server_High_Usage)

{'Server A':                               Server  CPU Usage  Memory Usage  Disk Usage
Timestamp                                                                
2026-03-02 14:20:03.188754  Server A         89           768          58
2026-03-02 14:20:03.217501  Server A         97          2560          84
2026-03-02 14:20:03.289799  Server A         84          1024          16
2026-03-02 14:20:03.301588  Server A         91          2560          57
2026-03-02 14:20:03.312252  Server A         99          2816          25
...                              ...        ...           ...         ...
2026-03-02 14:20:20.787073  Server A        100          3072          66
2026-03-02 14:20:20.810405  Server A         86          3328          46
2026-03-02 14:20:20.829279  Server A         81          1024          45
2026-03-02 14:20:20.835918  Server A         90          2816          26
2026-03-02 14:20:20.856775  Server A         93          1024          47

[675 rows x 4 columns], 

In [21]:
#Pringing data into each CSV file
Server_High_Usage.get("Server A").to_csv("Server_A_High_Usage.csv")
Server_High_Usage.get("Server B").to_csv("Server_B_High_Usage.csv")
Server_High_Usage.get("Server C").to_csv("Server_C_High_Usage.csv")

In [ ]:
#Working to find rolling averages and trends

#First, Confirming the data is in the correct format and order
#Then find data for only Server A
#print(df[df["Server"] == "Server A"])

# Calculate rolling average of CPU Usage for Server A with a window of 10 entries
Server_A_RAW = df[df["Server"] == "Server A"]
Server_A_Crash_Chance = Server_A_RAW["CPU Usage"].rolling(window=10).mean().dropna()
#print(Server_A_Crash_Chance)

#Print only when the rolling average exceeds 75%
Server_A_Crash_Chance_Large = Server_A_Crash_Chance[Server_A_Crash_Chance > 75]
#print("Server Crash Chance for Server A when rolling average exceeds 75%:")
#print(Server_A_Crash_Chance_Large)

#Find the first timestamp where the rolling average exceeds 75%
if not Server_A_Crash_Chance_Large.empty:
	Server_A_Crash_Loc = Server_A_Crash_Chance_Large.index[0]
	print("\nFirst timestamp where rolling average exceeds 75%:")
	print(Server_A_Crash_Loc)

	#Find out when Server_A_Crash_Chance_Large.index[0] = Server_A_Crash_Chance.index[x]
	#idx = Server_A_Crash_Chance.index.get_indexer([Server_A_Crash_Chance_Large.index[0]])
	#print("Index of the first timestamp where rolling average exceeds 75%:")
	#print(idx)

	# This doesn't work because the index is a datetime, not an integer. 
	# Due to timestamp being the index, I can't use .get_indexer() due to duplicate timestamps. 


	# Create a True/False series for when the average exceeds 75%
	is_crashing = Server_A_Crash_Chance > 75

	# .argmax() finds the integer row number of the very first 'True'
	idx = is_crashing.argmax() 

	print("\nInteger row number of the first spike:")
	print(idx)

	# Now that you have the exact row number, you can slice the 10 logs 
	# using .iloc (Integer Location)
	the_10_logs = Server_A_RAW.iloc[idx : idx + 9]
	#USED Gemini to debug -> because i used dropna() on the rolling average, 
	# I needed to push the idx by 10 to compensate for the dropped rows.

	print("\nThe exact 10 rolling averages leading to the spike:")
	print(the_10_logs)
else:
	print("No rolling average exceeds 75% for Server A.")


First timestamp where rolling average exceeds 75%:
2026-03-02 14:20:04.522484

Integer row number of the first spike:
249

The exact 10 rolling averages leading to the spike:
                              Server  CPU Usage  Memory Usage  Disk Usage
Timestamp                                                                
2026-03-02 14:20:04.480698  Server A         85           768          54
2026-03-02 14:20:04.482284  Server A         68          1280          47
2026-03-02 14:20:04.490347  Server A         81          2816          17
2026-03-02 14:20:04.493538  Server A         87          3072          69
2026-03-02 14:20:04.506554  Server A         77          3840          92
2026-03-02 14:20:04.511328  Server A         23          1792          22
2026-03-02 14:20:04.512914  Server A         99          2048          55
2026-03-02 14:20:04.514503  Server A         75          2816          37
2026-03-02 14:20:04.520904  Server A         70           512          99
